In [ ]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




In [2]:
#build the sensor to sensor networks for all time points. 

# ================================================================
# Sensor-to-sensor (station) network ONLY -- exact same construction
# approach as every model script so far (haversine distance, wind-
# alignment edge weights, hybrid close/far rule). 2016-2019, full
# dataset. Saved so every future script can just load this instead
# of recomputing it.
# ================================================================
import numpy as np
import pandas as pd
import json
import gc

BASE = "/Users/drewbaldwin/PM2_5 Research"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)
print(f"station graph: {n_stations} stations, {edge_index_np.shape[1]} directed edges")

dt_index = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
del df
gc.collect()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()

ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component, wind_component
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
del train_nonzero, station_wind_raw
gc.collect()

print(f"hourly station edge weight array: {station_wind_edge_weight.shape}  (n_time={n_time}, n_edges={edge_index_np.shape[1]})")
print(f"wind_scale_s (train-set std of nonzero weights, used for normalization): {wind_scale_s:.4f}")

np.savez_compressed(
    f"{BASE}/precomputed_station_graph.npz",
    edge_index=edge_index_np,
    station_wind_edge_weight=station_wind_edge_weight,
    lats=lats, lons=lons,
    split_id_per_hour=split_id_per_hour,
    wind_scale_s=wind_scale_s,
)

with open(f"{BASE}/precomputed_station_metadata.json", "w") as f:
    json.dump({
        "station_order": [str(s) for s in station_order],
        "n_stations": int(n_stations),
        "n_time": int(n_time),
        "n_edges": int(edge_index_np.shape[1]),
        "DIST_CUTOFF": DIST_CUTOFF, "RHO_KM": RHO_KM, "HYBRID_THRESHOLD_KM": HYBRID_THRESHOLD_KM,
    }, f, indent=2)

print(f"\nsaved precomputed_station_graph.npz and precomputed_station_metadata.json to {BASE}")


station graph: 176 stations, 21414 directed edges
hourly station edge weight array: (52608, 21414)  (n_time=52608, n_edges=21414)
wind_scale_s (train-set std of nonzero weights, used for normalization): 3.7875

saved precomputed_station_graph.npz and precomputed_station_metadata.json to /Users/drewbaldwin/PM2_5 Research


In [ ]:
# ================================================================
# REGION-DEFINITION SWEEP: same wind_graph staged model (seed=0,
# H=24), but the region "episode" label is built 3 different ways
# from the same underlying station-level data:
#   1) MEAN:  region-average PM2.5 >= 75 for >=2 consecutive hours  (current default)
#   2) PCT25: >=25% of stations in the region simultaneously >=75, for >=2h
#   3) PCT75: >=75% of stations in the region simultaneously >=75, for >=2h
#
# Stage A (trajectory regression) doesn't depend on the label
# definition at all -- it's trained ONCE and reused. Only Stage B
# (classification + bucket head) is retrained per definition, since
# that's the only place the label choice matters.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
SEED = 0
BUCKET_SPLIT = 12
BUCKET_WEIGHT = 0.3
MAX_EPOCHS_STAGE_A = 15
MAX_EPOCHS_STAGE_B = 15
PATIENCE = 5
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24
GRAPH_RECENT_HOURS = 18
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

# ---------------- THE THREE REGION-EVENT DEFINITIONS ----------------
def rolling_min_sustain(arr_2d, hours):
    rev = arr_2d[::-1]
    rm = pd.DataFrame(rev).rolling(window=hours, min_periods=hours).min().to_numpy()
    return rm[::-1]

# 1) MEAN-based (current default)
region_episode_label_MEAN = (rolling_min_sustain(region_pm25, SUSTAIN_HOURS) >= EVENT_THRESHOLD).astype(np.float32)

# 2) & 3) fraction-of-stations-exceeding based
exceed_station = (pm25_raw_arr >= EVENT_THRESHOLD).astype(np.float32)  # (n_time, n_stations)
frac_exceed = np.zeros((n_time, n_regions), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    frac_exceed[:, r] = exceed_station[:, cols].mean(axis=1)
frac_exceed_sustained = rolling_min_sustain(frac_exceed, SUSTAIN_HOURS)
region_episode_label_PCT25 = (frac_exceed_sustained >= 0.25).astype(np.float32)
region_episode_label_PCT75 = (frac_exceed_sustained >= 0.75).astype(np.float32)

LABEL_DEFS = {
    "MEAN": region_episode_label_MEAN,
    "PCT25": region_episode_label_PCT25,
    "PCT75": region_episode_label_PCT75,
}
for name, lab in LABEL_DEFS.items():
    rate = np.nanmean(lab[TRAIN_MASK])
    print(f"definition {name}: train-set positive rate = {rate*100:.3f}%")

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class TrajectoryBackbone(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.rmem = region_membership_t_local
        self.horizon = horizon
        self.gru_hidden = gru_hidden

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        return dec_out.reshape(B, n_reg, self.horizon, self.gru_hidden)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1 = 2
print(f"\ndevice={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

reg_pm25_mean = np.nanmean(region_pm25[TRAIN_MASK])
reg_pm25_std = np.nanstd(region_pm25[TRAIN_MASK]) + 1e-6
region_pm25_std_arr = np.nan_to_num((region_pm25 - reg_pm25_mean) / reg_pm25_std, nan=0.0).astype(np.float32)

# build windows ONCE; collect all 3 label trajectories per window (cheap slicing)
p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    Xl, pmtl, sl, labd = p_buckets[s_start]
    Xl.append(time_arr_std[t:t + WINDOW])
    pmtl.append(region_pm25_std_arr[t + WINDOW: t + WINDOW + HORIZON].T)
    sl.append(t)
    labd.append({name: lab[t + WINDOW: t + WINDOW + HORIZON].T for name, lab in LABEL_DEFS.items()})

def assemble(bucket):
    Xl, pmtl, sl, labd = bucket
    X = np.stack(Xl); pmt = np.stack(pmtl); starts = np.stack(sl)
    labs = {name: np.stack([d[name] for d in labd]) for name in LABEL_DEFS}
    return X, pmt, starts, labs

X0, pmtraj0, starts0, ytraj0_defs = assemble(p_buckets[0])
X1, pmtraj1, starts1, ytraj1_defs = assemble(p_buckets[1])
X2, pmtraj2, starts2, ytraj2_defs = assemble(p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

already_ongoing_defs = {}
for name, lab in LABEL_DEFS.items():
    already_ongoing_defs[name] = {
        0: torch.tensor(lab[starts0 + WINDOW - 1, :] > 0),
        1: torch.tensor(lab[starts1 + WINDOW - 1, :] > 0),
        2: (lab[starts2 + WINDOW - 1, :] > 0),
    }

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

pmtraj0_t = torch.tensor(pmtraj0, dtype=torch.float32)
pmtraj1_t = torch.tensor(pmtraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))

ytraj_t = {0: {}, 1: {}}
for name in LABEL_DEFS:
    ytraj_t[0][name] = torch.tensor(ytraj0_defs[name], dtype=torch.float32)
    ytraj_t[1][name] = torch.tensor(ytraj1_defs[name], dtype=torch.float32)
gc.collect()

def run_p1_epoch_graph(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

print("\n=== Phase 1: pretraining station encoder (wind_graph, seed=0) ===")
torch.manual_seed(SEED); np.random.seed(SEED)
lr, dropout, wd = 0.003, 0.5, 0.0005
# Phase 1 still needs a per-station next-hour target; reuse yreg-style target quickly
yreg0 = time_arr_std[starts0 + WINDOW, :, pm25_col_idx] if False else None  # not needed; use pmtraj-independent quantile pretrain on station PM25 next-step
# (kept simple: reuse pinball regression target identical to earlier scripts)
target_idx0 = starts0 + WINDOW
target_idx1 = starts1 + WINDOW
yreg0_t = torch.tensor(((pm25_raw_arr[target_idx0, :] - t_mean[0,0,pm25_col_idx]) / t_std[0,0,pm25_col_idx]), dtype=torch.float32)
yreg1_t = torch.tensor(((pm25_raw_arr[target_idx1, :] - t_mean[0,0,pm25_col_idx]) / t_std[0,0,pm25_col_idx]), dtype=torch.float32)

p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=wd)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch_graph(p1, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch_graph(p1, X1_t, yreg1_t, starts1_t, opt1, False)
    if vl < best_p1_val: best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
station_encoder = copy.deepcopy(p1.station_conv); del p1; gc.collect()

print("\n=== Stage A: trajectory-regression-only backbone (shared across all 3 definitions) ===")
torch.manual_seed(SEED); np.random.seed(SEED)
backbone = TrajectoryBackbone(copy.deepcopy(station_encoder), region_membership_t, dropout).to(DEVICE)
reg_head = nn.Linear(backbone.gru_hidden, N_QUANTILES).to(DEVICE)
stageA_params = list(backbone.parameters()) + list(reg_head.parameters())
optA = torch.optim.Adam(stageA_params, lr=lr, weight_decay=wd)

def run_stageA_epoch(X, pm_traj, starts, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    backbone.train(train); reg_head.train(train)
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    total_loss, total_n = 0.0, 0
    for start in range(0, n, eff_batch):
        if train: optA.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            pmb = pm_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                dec_out = backbone(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                reg_quantiles = monotonic_quantiles(reg_head(dec_out))
                loss = pinball_loss(reg_quantiles, pmb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optA.step()
    return total_loss / total_n

best_vlA, best_stateA, patience_ctr, best_epochA = float("inf"), None, 0, 0
for epoch in range(1, MAX_EPOCHS_STAGE_A + 1):
    tl = run_stageA_epoch(X0_t, pmtraj0_t, starts0_t, True)
    vl = run_stageA_epoch(X1_t, pmtraj1_t, starts1_t, False)
    if vl < best_vlA:
        best_vlA, best_stateA, patience_ctr, best_epochA = vl, (copy.deepcopy(backbone.state_dict()), copy.deepcopy(reg_head.state_dict())), 0, epoch
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE: break
backbone.load_state_dict(best_stateA[0]); reg_head.load_state_dict(best_stateA[1])
print(f"Stage A best_epoch={best_epochA} val_pinball={best_vlA:.4f}")
for p in backbone.parameters(): p.requires_grad_(False)
backbone.eval()

def bucket_loss_from_logits(logits, yb, ao):
    B_, R_, H_ = logits.shape
    logits_flat = logits.reshape(B_ * R_, H_)
    yb_flat = yb.reshape(B_ * R_, H_)
    ao_flat = ao.reshape(B_ * R_)
    has_ep = yb_flat.max(dim=-1).values > 0
    target_hour_all = yb_flat.argmax(dim=-1)
    is_continuing = (target_hour_all == 0) & ao_flat
    valid = has_ep & (~is_continuing)
    if valid.any():
        bucket_target = (target_hour_all[valid] >= BUCKET_SPLIT).long()
        early_logit = logits_flat[valid, :BUCKET_SPLIT].max(dim=-1).values
        late_logit = logits_flat[valid, BUCKET_SPLIT:].max(dim=-1).values
        bucket_logits = torch.stack([early_logit, late_logit], dim=-1)
        bl = F.cross_entropy(bucket_logits, bucket_target)
    else:
        bl = torch.tensor(0.0, device=DEVICE)
    return bl

def predict_logits(X, starts, batch_size=64):
    backbone.eval(); cls_head.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            dec_out = backbone(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            cls_logits = cls_head(dec_out).squeeze(-1)
            out[idx.numpy()] = cls_logits.cpu().numpy()
    return out

def naive_bucket_pred(prob):
    return 1 if prob[BUCKET_SPLIT:].max() > prob[:BUCKET_SPLIT].max() else 0

def bucket_accuracy(prob_arr, ytraj_test, already_ongoing_test):
    correct, total = 0, 0
    for w in range(ytraj_test.shape[0]):
        for r in range(n_regions):
            lab = ytraj_test[w, r, :]
            if lab.max() == 0: continue
            onset_h = int(np.argmax(lab))
            if onset_h == 0 and already_ongoing_test[w, r]: continue
            true_b = 1 if onset_h >= BUCKET_SPLIT else 0
            pred_b = naive_bucket_pred(prob_arr[w, r, :])
            correct += int(pred_b == true_b); total += 1
    return correct / max(total, 1), total

all_results = {}
for name in LABEL_DEFS:
    print(f"\n=== Stage B [{name}]: classification + bucket head on frozen backbone ===")
    torch.manual_seed(SEED); np.random.seed(SEED)
    cls_head = nn.Linear(backbone.gru_hidden, 1).to(DEVICE)
    optB = torch.optim.Adam(cls_head.parameters(), lr=0.003, weight_decay=0.0005)

    ytraj0_n, ytraj1_n = ytraj_t[0][name], ytraj_t[1][name]
    ao0_n, ao1_n = already_ongoing_defs[name][0], already_ongoing_defs[name][1]
    pos_weight = min(float((ytraj0_n.numel() - ytraj0_n.sum()) / ytraj0_n.sum().clamp(min=1)), 50.0)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    print(f"  pos_weight={pos_weight:.2f}")

    def run_epoch(X, y_traj, starts, ao, train):
        n = X.shape[0]
        idx = torch.randperm(n) if train else torch.arange(n)
        cls_head.train(train)
        eff_batch = MICRO_BATCH * ACCUM_STEPS
        for start in range(0, n, eff_batch):
            if train: optB.zero_grad()
            batch_idx = idx[start:start + eff_batch]
            for ms in range(0, len(batch_idx), MICRO_BATCH):
                mb_idx = batch_idx[ms:ms + MICRO_BATCH]
                if len(mb_idx) == 0: continue
                xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
                yb = y_traj[mb_idx].to(DEVICE)
                aob = ao[mb_idx].to(DEVICE)
                ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
                ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
                with torch.no_grad():
                    dec_out = backbone(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                with torch.set_grad_enabled(train):
                    cls_logits = cls_head(dec_out).squeeze(-1)
                    bce = criterion(cls_logits, yb)
                    bl = bucket_loss_from_logits(cls_logits, yb, aob)
                    loss = bce + BUCKET_WEIGHT * bl
                if train: loss.backward()
            if train: optB.step()

    y_val_agg = ytraj1_n.numpy().max(axis=-1).reshape(-1)
    best_va, best_state, patience_ctr, best_epoch = -1.0, None, 0, 0
    for epoch in range(1, MAX_EPOCHS_STAGE_B + 1):
        run_epoch(X0_t, ytraj0_n, starts0_t, ao0_n, True)
        run_epoch(X1_t, ytraj1_n, starts1_t, ao1_n, False)
        logits1 = predict_logits(X1_t, starts1_t)
        va = average_precision_score(y_val_agg, (1/(1+np.exp(-logits1))).max(axis=-1).reshape(-1))
        if va > best_va: best_va, best_state, patience_ctr, best_epoch = va, copy.deepcopy(cls_head.state_dict()), 0, epoch
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE: break
    cls_head.load_state_dict(best_state)
    print(f"  best_epoch={best_epoch} val_aucpr={best_va:.4f}")

    logits2 = predict_logits(X2_t, starts2_t)
    prob2 = 1 / (1 + np.exp(-logits2))
    ytraj2_n = ytraj2_defs[name]
    ao2_n = already_ongoing_defs[name][2]
    y_test_agg = ytraj2_n.max(axis=-1).reshape(-1)
    score_test = prob2.max(axis=-1).reshape(-1)
    aucroc = roc_auc_score(y_test_agg, score_test)
    aucpr = average_precision_score(y_test_agg, score_test)
    bacc, n_bucket = bucket_accuracy(prob2, ytraj2_n, ao2_n)

    print(f"  test AUC-ROC={aucroc:.4f}  AUC-PR={aucpr:.4f}  bucket_acc={bacc:.4f} (n={n_bucket})")
    all_results[name] = {"aucroc": float(aucroc), "aucpr": float(aucpr), "bucket_acc": float(bacc), "n_bucket": int(n_bucket), "best_epoch": int(best_epoch)}

print(f"\n{'='*20} REGION-DEFINITION SWEEP SUMMARY {'='*20}")
print(f"{'definition':10s} {'AUC-ROC':>10s} {'AUC-PR':>10s} {'bucket_acc':>12s} {'n_bucket':>10s}")
for name, r in all_results.items():
    print(f"{name:10s} {r['aucroc']:>10.4f} {r['aucpr']:>10.4f} {r['bucket_acc']:>12.4f} {r['n_bucket']:>10d}")

with open(f"{BASE}/region_definition_sweep_2019_wind_graph_seed0.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nsaved to {BASE}/region_definition_sweep_2019_wind_graph_seed0.json")


In [2]:
# ================================================================
# WEIBULL TIME-TO-EVENT HEAD: predicts a continuous distribution
# (Weibull shape k, scale lambda) over "hours until onset," directly
# off h_final -- NOT the 24-step decoder. Trained via proper censored
# log-likelihood: density at the true onset hour for fresh episodes,
# survival-past-24 for windows with no episode at all. "Continuing"
# episodes excluded (same reasoning as every other timing approach
# today -- left-truncated, no clean onset to model).
#
# This bypasses the decoder/bucket/hazard mechanisms entirely. Only
# 2 numbers come out per window/region, which should be far more
# stable than the 24 independently-noisy hazard values that failed
# earlier today.
#
# wind_graph, seed=0, H=24, MEAN region-event definition (current
# default) -- same backbone-pretraining pattern as the staged model.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json

BASE = "/Users/drewbaldwin/PM2_5 Research"
SEED = 0
BUCKET_SPLIT = 12
MAX_EPOCHS_STAGE_A = 15
MAX_EPOCHS_WEIBULL = 20
PATIENCE = 5
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24
GRAPH_RECENT_HOURS = 18
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

reg_pm25_mean = np.nanmean(region_pm25[TRAIN_MASK])
reg_pm25_std = np.nanstd(region_pm25[TRAIN_MASK]) + 1e-6
region_pm25_std_arr = np.nan_to_num((region_pm25 - reg_pm25_mean) / reg_pm25_std, nan=0.0).astype(np.float32)

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class TrajectoryBackbone(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.rmem = region_membership_t_local
        self.horizon = horizon
        self.gru_hidden = gru_hidden

    def _encode_raw(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        """ Returns h_final in GRU-native form: (1, B*n_reg, gru_hidden) -- the correct
            shape to feed straight back into decoder_gru as its initial hidden state. """
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final_raw = self.region_gru(h_region_seq)  # (1, B*n_reg, gru_hidden)
        return h_final_raw, B

    def encode(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        """ Returns h_final as (B, n_reg, gru_hidden) -- for external heads (e.g. WeibullHead). """
        h_final_raw, B = self._encode_raw(x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg)
        return h_final_raw.squeeze(0).reshape(B, n_reg, self.gru_hidden)

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        """ Full forward incl. decoder -- used only by Stage A (regression pretraining). """
        h_final_raw, B = self._encode_raw(x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg)
        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final_raw)  # h_final_raw already correct 3D shape, no reshape needed
        dec_out = self.drop(dec_out)
        return dec_out.reshape(B, n_reg, self.horizon, self.gru_hidden)

class WeibullHead(nn.Module):
    def __init__(self, gru_hidden=32):
        super().__init__()
        self.head = nn.Linear(gru_hidden, 2)

    def forward(self, h_final):
        raw = self.head(h_final)
        k = F.softplus(raw[..., 0]) + 0.05
        lam = F.softplus(raw[..., 1]) + 0.5
        return k, lam

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1 = 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    pm25_traj = region_pm25_std_arr[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, pmtrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    pmtrajl.append(pm25_traj.T)
    sl.append(t)

X0, yreg0, ytraj0, pmtraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, pmtraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, pmtraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

already_ongoing0 = (region_episode_label[starts0 + WINDOW - 1, :] > 0)
already_ongoing1 = (region_episode_label[starts1 + WINDOW - 1, :] > 0)
already_ongoing2 = (region_episode_label[starts2 + WINDOW - 1, :] > 0)
already_ongoing0_t = torch.tensor(already_ongoing0)
already_ongoing1_t = torch.tensor(already_ongoing1)

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
pmtraj0_t = torch.tensor(pmtraj0, dtype=torch.float32)
pmtraj1_t = torch.tensor(pmtraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

def run_p1_epoch_graph(model, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

print("\n=== Phase 1: pretraining station encoder (wind_graph, seed=0) ===")
torch.manual_seed(SEED); np.random.seed(SEED)
lr, dropout, wd = 0.003, 0.5, 0.0005
p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=wd)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch_graph(p1, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch_graph(p1, X1_t, yreg1_t, starts1_t, opt1, False)
    if vl < best_p1_val: best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
station_encoder = copy.deepcopy(p1.station_conv); del p1; gc.collect()

print("\n=== Stage A: trajectory-regression-only backbone (shared encoder) ===")
torch.manual_seed(SEED); np.random.seed(SEED)
backbone = TrajectoryBackbone(copy.deepcopy(station_encoder), region_membership_t, dropout).to(DEVICE)
reg_head = nn.Linear(backbone.gru_hidden, N_QUANTILES).to(DEVICE)
stageA_params = list(backbone.parameters()) + list(reg_head.parameters())
optA = torch.optim.Adam(stageA_params, lr=lr, weight_decay=wd)

def run_stageA_epoch(X, pm_traj, starts, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    backbone.train(train); reg_head.train(train)
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    total_loss, total_n = 0.0, 0
    for start in range(0, n, eff_batch):
        if train: optA.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            pmb = pm_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                dec_out = backbone(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                reg_quantiles = monotonic_quantiles(reg_head(dec_out))
                loss = pinball_loss(reg_quantiles, pmb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optA.step()
    return total_loss / total_n

best_vlA, best_stateA, patience_ctr, best_epochA = float("inf"), None, 0, 0
for epoch in range(1, MAX_EPOCHS_STAGE_A + 1):
    tl = run_stageA_epoch(X0_t, pmtraj0_t, starts0_t, True)
    vl = run_stageA_epoch(X1_t, pmtraj1_t, starts1_t, False)
    if vl < best_vlA:
        best_vlA, best_stateA, patience_ctr, best_epochA = vl, copy.deepcopy(backbone.state_dict()), 0, epoch
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE: break
backbone.load_state_dict(best_stateA)
print(f"Stage A best_epoch={best_epochA} val_pinball={best_vlA:.4f}")
for p in backbone.parameters(): p.requires_grad_(False)
backbone.eval()

print("\n=== Weibull head: censored time-to-event training on frozen h_final ===")
torch.manual_seed(SEED); np.random.seed(SEED)
weibull_head = WeibullHead(backbone.gru_hidden).to(DEVICE)
optW = torch.optim.Adam(weibull_head.parameters(), lr=0.01, weight_decay=0.0005)

def weibull_nll(k, lam, yb, ao):
    B_, R_ = k.shape
    yb_flat = yb.reshape(B_ * R_, HORIZON) if yb.dim() == 3 else yb
    ao_flat = ao.reshape(-1)
    k_flat, lam_flat = k.reshape(-1), lam.reshape(-1)
    has_ep = yb_flat.max(dim=-1).values > 0
    onset_h = yb_flat.argmax(dim=-1)
    is_continuing = (onset_h == 0) & ao_flat
    valid = ~is_continuing
    if not valid.any():
        return torch.tensor(0.0, device=k.device)
    k_v, lam_v = k_flat[valid], lam_flat[valid]
    has_ep_v = has_ep[valid]
    t_used = torch.where(has_ep_v, onset_h[valid].float() + 0.5, torch.full_like(onset_h[valid].float(), float(HORIZON)))
    ratio = (t_used / lam_v).clamp(min=1e-6)
    pow_term = ratio ** k_v
    log_dens_extra = torch.log(k_v) - torch.log(lam_v) + (k_v - 1) * torch.log(ratio)
    nll = pow_term - has_ep_v.float() * log_dens_extra
    return nll.mean()

def run_weibull_epoch(X, y_traj, starts, ao, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    weibull_head.train(train)
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    total_loss, total_n = 0.0, 0
    for start in range(0, n, eff_batch):
        if train: optW.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            aob = ao[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.no_grad():
                h_final = backbone.encode(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            with torch.set_grad_enabled(train):
                k, lam = weibull_head(h_final)
                loss = weibull_nll(k, lam, yb, aob)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optW.step()
    return total_loss / total_n

best_vlW, best_stateW, patience_ctr, best_epochW = float("inf"), None, 0, 0
for epoch in range(1, MAX_EPOCHS_WEIBULL + 1):
    tl = run_weibull_epoch(X0_t, ytraj0_t, starts0_t, already_ongoing0_t, True)
    vl = run_weibull_epoch(X1_t, ytraj1_t, starts1_t, already_ongoing1_t, False)
    print(f"  epoch {epoch}  train_nll={tl:.4f}  val_nll={vl:.4f}")
    if vl < best_vlW: best_vlW, best_stateW, patience_ctr, best_epochW = vl, copy.deepcopy(weibull_head.state_dict()), 0, epoch
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE: break
weibull_head.load_state_dict(best_stateW)
print(f"best_epoch={best_epochW} val_nll={best_vlW:.4f}")

# ================================================================
# EVALUATE: predicted Weibull mean/median vs true onset hour
# ================================================================
print(f"\n{'='*20} Weibull timing evaluation {'='*20}")

def predict_weibull_params(X, starts, batch_size=64):
    backbone.eval(); weibull_head.eval()
    n = X.shape[0]
    k_out = np.zeros((n, n_regions), dtype=np.float32)
    lam_out = np.zeros((n, n_regions), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            h_final = backbone.encode(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            k, lam = weibull_head(h_final)
            k_out[idx.numpy()] = k.cpu().numpy()
            lam_out[idx.numpy()] = lam.cpu().numpy()
    return k_out, lam_out

k_test, lam_test = predict_weibull_params(X2_t, starts2_t)

from scipy.special import gammaln
weibull_mean = lam_test * np.exp(gammaln(1 + 1.0 / k_test))
weibull_median = lam_test * (np.log(2) ** (1.0 / k_test))

true_onsets, pred_mean, pred_median = [], [], []
for w in range(ytraj2.shape[0]):
    for r in range(n_regions):
        lab = ytraj2[w, r, :]
        if lab.max() == 0: continue
        onset_h = int(np.argmax(lab))
        if onset_h == 0 and already_ongoing2[w, r]: continue
        true_onsets.append(onset_h)
        pred_mean.append(weibull_mean[w, r])
        pred_median.append(weibull_median[w, r])

true_onsets = np.array(true_onsets)
pred_mean = np.clip(np.array(pred_mean), 0, HORIZON - 1)
pred_median = np.clip(np.array(pred_median), 0, HORIZON - 1)

mae_mean = np.mean(np.abs(true_onsets - pred_mean))
mae_median = np.mean(np.abs(true_onsets - pred_median))

true_buckets = (true_onsets >= BUCKET_SPLIT).astype(int)
bucket_from_mean = (pred_mean >= BUCKET_SPLIT).astype(int)
bucket_from_median = (pred_median >= BUCKET_SPLIT).astype(int)
bacc_mean = (bucket_from_mean == true_buckets).mean()
bacc_median = (bucket_from_median == true_buckets).mean()

print(f"n valid (fresh-onset) episode cases: {len(true_onsets)}")
print(f"mean predicted k (shape): {k_test.mean():.3f}   mean predicted lambda (scale): {lam_test.mean():.3f}")
print(f"\nMAE using Weibull MEAN as point estimate:   {mae_mean:.3f} hours   (bucket_acc={bacc_mean:.4f})")
print(f"MAE using Weibull MEDIAN as point estimate: {mae_median:.3f} hours   (bucket_acc={bacc_median:.4f})")
print(f"\nfor comparison:")
print(f"  naive (always predict mean onset=10.3h):     MAE = 6.054 hours")
print(f"  bucket model (midpoint translation):          MAE = 5.357 hours  (bucket_acc=0.706-0.713)")
print(f"  hazard model (argmax / expected value):       MAE = 6.309 / 9.631 hours")

results = {
    "val_nll": float(best_vlW), "best_epoch": int(best_epochW),
    "mean_k": float(k_test.mean()), "mean_lambda": float(lam_test.mean()),
    "mae_weibull_mean": float(mae_mean), "bucket_acc_from_mean": float(bacc_mean),
    "mae_weibull_median": float(mae_median), "bucket_acc_from_median": float(bacc_median),
    "n_valid": int(len(true_onsets)),
}
with open(f"{BASE}/weibull_timing_2019_wind_graph_seed0.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nsaved to {BASE}/weibull_timing_2019_wind_graph_seed0.json")


device=mps
windows: train=17485 val=8701 test=8701

=== Phase 1: pretraining station encoder (wind_graph, seed=0) ===

=== Stage A: trajectory-regression-only backbone (shared encoder) ===
Stage A best_epoch=12 val_pinball=0.1268

=== Weibull head: censored time-to-event training on frozen h_final ===
  epoch 1  train_nll=2.4822  val_nll=1.5362
  epoch 2  train_nll=1.4129  val_nll=1.3400
  epoch 3  train_nll=1.2962  val_nll=1.2740
  epoch 4  train_nll=1.2478  val_nll=1.2404
  epoch 5  train_nll=1.2206  val_nll=1.2198
  epoch 6  train_nll=1.2029  val_nll=1.2056
  epoch 7  train_nll=1.1900  val_nll=1.1948
  epoch 8  train_nll=1.1799  val_nll=1.1862
  epoch 9  train_nll=1.1717  val_nll=1.1790
  epoch 10  train_nll=1.1647  val_nll=1.1728
  epoch 11  train_nll=1.1585  val_nll=1.1672
  epoch 12  train_nll=1.1531  val_nll=1.1624
  epoch 13  train_nll=1.1482  val_nll=1.1579
  epoch 14  train_nll=1.1437  val_nll=1.1534
  epoch 15  train_nll=1.1394  val_nll=1.1494
  epoch 16  train_nll=1.1353  v

In [1]:
# ================================================================
# SENSOR-LEVEL 24h classification: wind_graph vs minimal_gru (no
# graph). Same decoder+bucket-loss pattern already validated at the
# region level, now applied PER STATION -- no region aggregation,
# no region graph, no ambiguous "region trajectory" definition at
# all. This directly tests whether the graph conv structure helps
# at the sensor level the way it did at the region level, and
# whether station-level classification extends cleanly from the
# already-strong +1h result out to the full 24h horizon.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
SEED = 0
BUCKET_SPLIT = 12
BUCKET_WEIGHT = 0.3
MAX_EPOCHS_P1 = 2
MAX_EPOCHS_P2 = 15
PATIENCE = 5
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
WINDOW = 36
HORIZON = 24
GRAPH_RECENT_HOURS = 18
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

# ---------------- STATION-level episode label (no region aggregation at all) ----------------
rev = pm25_raw_arr[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
station_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()
print(f"station-level episode rate (train): {np.nanmean(station_episode_label[TRAIN_MASK])*100:.3f}%")

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class StationDecoderGCN(nn.Module):
    """ wind_graph: station graph conv (per hour) -> GRU -> decoder GRU -> 24 classification logits, per station. """
    def __init__(self, station_conv, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.drop = nn.Dropout(dropout)
        self.station_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.output_head = nn.Linear(gru_hidden, 1)
        self.horizon = horizon

    def forward(self, x_window, edge_index, edge_weight_seq, n_stat):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.station_gru(h_seq)  # (1, B*N, gru_hidden)
        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * N, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)
        return logits.reshape(B, n_stat, self.horizon)

class StationDecoderGRU(nn.Module):
    """ minimal_gru: NO graph conv at all, just a linear station projection. """
    def __init__(self, station_lin, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_lin = station_lin
        self.drop = nn.Dropout(dropout)
        self.station_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.output_head = nn.Linear(gru_hidden, 1)
        self.horizon = horizon

    def forward(self, x_window, n_stat):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h = self.drop(h)
            h_seq.append(h)
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.station_gru(h_seq)
        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * N, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)
        return logits.reshape(B, n_stat, self.horizon)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
print(f"device={DEVICE}")

edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
del train_nonzero, station_wind_raw
gc.collect()
wind_ewt_s = torch.tensor(station_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(time_arr_std[t:t + WINDOW])
    yregl.append(time_arr_std[t + WINDOW, :, pm25_col_idx])
    ytrajl.append(station_episode_label[t + WINDOW: t + WINDOW + HORIZON].T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

already_ongoing0 = (station_episode_label[starts0 + WINDOW - 1, :] > 0)
already_ongoing1 = (station_episode_label[starts1 + WINDOW - 1, :] > 0)
already_ongoing2 = (station_episode_label[starts2 + WINDOW - 1, :] > 0)
already_ongoing0_t = torch.tensor(already_ongoing0)
already_ongoing1_t = torch.tensor(already_ongoing1)

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def bucket_loss_from_logits(logits, yb, ao):
    B_, R_, H_ = logits.shape
    logits_flat = logits.reshape(B_ * R_, H_)
    yb_flat = yb.reshape(B_ * R_, H_)
    ao_flat = ao.reshape(B_ * R_)
    has_ep = yb_flat.max(dim=-1).values > 0
    target_hour_all = yb_flat.argmax(dim=-1)
    is_continuing = (target_hour_all == 0) & ao_flat
    valid = has_ep & (~is_continuing)
    if valid.any():
        bucket_target = (target_hour_all[valid] >= BUCKET_SPLIT).long()
        early_logit = logits_flat[valid, :BUCKET_SPLIT].max(dim=-1).values
        late_logit = logits_flat[valid, BUCKET_SPLIT:].max(dim=-1).values
        bucket_logits = torch.stack([early_logit, late_logit], dim=-1)
        bl = F.cross_entropy(bucket_logits, bucket_target)
    else:
        bl = torch.tensor(0.0, device=DEVICE)
    return bl

def run_p2_epoch_graph(model, ewt, X, y_traj, starts, already_ongoing, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ao = already_ongoing[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_seq, n_stations)
                bce = region_criterion(logits, yb)
                bl = bucket_loss_from_logits(logits, yb, ao)
                loss = bce + BUCKET_WEIGHT * bl
            if train: loss.backward()
        if train: optimizer.step()

def run_p2_epoch_gru(model, X, y_traj, starts, already_ongoing, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ao = already_ongoing[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_stations)
                bce = region_criterion(logits, yb)
                bl = bucket_loss_from_logits(logits, yb, ao)
                loss = bce + BUCKET_WEIGHT * bl
            if train: loss.backward()
        if train: optimizer.step()

def predict_probs_graph(model, ewt, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_stations, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_seq = gather_seq(ewt, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_seq, n_stations)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_stations, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_stations)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def naive_bucket_pred(prob):
    return 1 if prob[BUCKET_SPLIT:].max() > prob[:BUCKET_SPLIT].max() else 0

def bucket_accuracy(tp):
    correct, total = 0, 0
    for w in range(ytraj2.shape[0]):
        for s in range(n_stations):
            lab = ytraj2[w, s, :]
            if lab.max() == 0: continue
            onset_h = int(np.argmax(lab))
            if onset_h == 0 and already_ongoing2[w, s]: continue
            true_b = 1 if onset_h >= BUCKET_SPLIT else 0
            pred_b = naive_bucket_pred(tp[w, s, :])
            correct += int(pred_b == true_b); total += 1
    return correct / max(total, 1), total

y_test_agg = ytraj2.max(axis=-1).reshape(-1)

def train_wind_graph(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    lr, dropout, wd = 0.003, 0.5, 0.0005
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=wd)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val: best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    enc = copy.deepcopy(p1.station_conv); del p1; gc.collect()
    model = StationDecoderGCN(enc, dropout).to(DEVICE)
    opt2 = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_va, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(model, wind_ewt_s, X0_t, ytraj0_t, starts0_t, already_ongoing0_t, opt2, True)
        run_p2_epoch_graph(model, wind_ewt_s, X1_t, ytraj1_t, starts1_t, already_ongoing1_t, opt2, False)
        vp = predict_probs_graph(model, wind_ewt_s, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
        print(f"  wind_graph epoch {epoch} val_aucpr={va:.4f}")
        if va > best_va: best_va, best_state, patience_ctr = va, copy.deepcopy(model.state_dict()), 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    tp = predict_probs_graph(model, wind_ewt_s, X2_t, starts2_t)
    del model; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return tp

def train_minimal_gru(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    lr, dropout, wd = 0.003, 0.3, 0.0005
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=wd)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val: best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    enc = copy.deepcopy(p1.station_lin); del p1; gc.collect()
    model = StationDecoderGRU(enc, dropout).to(DEVICE)
    opt2 = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_va, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(model, X0_t, ytraj0_t, starts0_t, already_ongoing0_t, opt2, True)
        run_p2_epoch_gru(model, X1_t, ytraj1_t, starts1_t, already_ongoing1_t, opt2, False)
        vp = predict_probs_gru(model, X1_t)
        va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
        print(f"  minimal_gru epoch {epoch} val_aucpr={va:.4f}")
        if va > best_va: best_va, best_state, patience_ctr = va, copy.deepcopy(model.state_dict()), 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    tp = predict_probs_gru(model, X2_t)
    del model; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    return tp

print("\n=== training wind_graph (station-level, H=24) ===")
tp_wind = train_wind_graph(SEED)
print("\n=== training minimal_gru (station-level, H=24) ===")
tp_gru = train_minimal_gru(SEED)

results = {}
for name, tp in [("wind_graph", tp_wind), ("minimal_gru", tp_gru)]:
    score = tp.max(axis=-1).reshape(-1)
    aucroc = roc_auc_score(y_test_agg, score)
    aucpr = average_precision_score(y_test_agg, score)
    bacc, n_bucket = bucket_accuracy(tp)
    results[name] = {"aucroc": float(aucroc), "aucpr": float(aucpr), "bucket_acc": float(bacc), "n_bucket": int(n_bucket)}
    print(f"{name:12s}  AUC-ROC={aucroc:.4f}  AUC-PR={aucpr:.4f}  bucket_acc={bacc:.4f} (n={n_bucket})")

print(f"\n{'='*20} STATION-LEVEL 24h: wind_graph vs minimal_gru {'='*20}")
print(f"{'metric':15s} {'wind_graph':>12s} {'minimal_gru':>12s}")
print(f"{'AUC-PR':15s} {results['wind_graph']['aucpr']:>12.4f} {results['minimal_gru']['aucpr']:>12.4f}")
print(f"{'bucket_acc':15s} {results['wind_graph']['bucket_acc']:>12.4f} {results['minimal_gru']['bucket_acc']:>12.4f}")

with open(f"{BASE}/station_level_24h_wind_vs_gru_seed0.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nsaved to {BASE}/station_level_24h_wind_vs_gru_seed0.json")


station-level episode rate (train): 1.160%
device=mps
windows: train=17485 val=8701 test=8701
pos_weight: 50.00

=== training wind_graph (station-level, H=24) ===
  wind_graph epoch 1 val_aucpr=0.5815
  wind_graph epoch 2 val_aucpr=0.5842
  wind_graph epoch 3 val_aucpr=0.5800
  wind_graph epoch 4 val_aucpr=0.5746
  wind_graph epoch 5 val_aucpr=0.5625
  wind_graph epoch 6 val_aucpr=0.5494
  wind_graph epoch 7 val_aucpr=0.5631

=== training minimal_gru (station-level, H=24) ===
  minimal_gru epoch 1 val_aucpr=0.5680
  minimal_gru epoch 2 val_aucpr=0.5706
  minimal_gru epoch 3 val_aucpr=0.5606
  minimal_gru epoch 4 val_aucpr=0.5664
  minimal_gru epoch 5 val_aucpr=0.5653
  minimal_gru epoch 6 val_aucpr=0.5578
  minimal_gru epoch 7 val_aucpr=0.5634
wind_graph    AUC-ROC=0.9605  AUC-PR=0.7393  bucket_acc=0.6987 (n=61797)
minimal_gru   AUC-ROC=0.9535  AUC-PR=0.7139  bucket_acc=0.6843 (n=61797)

==================== STATION-LEVEL 24h: wind_graph vs minimal_gru ====================
metric      

In [2]:
from sklearn.metrics import precision_recall_curve

for name, tp in [("wind_graph", tp_wind), ("minimal_gru", tp_gru)]:
    score = tp.max(axis=-1).reshape(-1)
    prec, rec, thresh = precision_recall_curve(y_test_agg, score)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    idx = np.nanargmax(f1[:-1])
    pred = (score >= thresh[idx]).astype(int)
    TP = int(((pred == 1) & (y_test_agg == 1)).sum())
    FP = int(((pred == 1) & (y_test_agg == 0)).sum())
    FN = int(((pred == 0) & (y_test_agg == 1)).sum())
    print(f"{name:12s}  threshold={thresh[idx]:.4f}  recall={rec[idx]*100:.1f}%  precision={prec[idx]*100:.1f}%  "
          f"({TP}/{TP+FN} detected, {FP} false alarms)")


wind_graph    threshold=0.8004  recall=65.2%  precision=68.6%  (58161/89260 detected, 26673 false alarms)
minimal_gru   threshold=0.8393  recall=62.4%  precision=67.1%  (55666/89260 detected, 27330 false alarms)


In [ ]:
# ================================================================
# STATION-LEVEL +1h severe-event comparison vs TransNet/AGATNet.
# Leak-controlled version:
#   * Loads the causal imputed file with BLH (inputs use only past or
#     same-hour measured data; imputed values are labeled by source).
#   * Phase 1 training loss is masked to station-hours whose PM2.5
#     target was actually observed (PM25_src == "observed").
#   * Threshold selection (validation) and all test metrics use only
#     observed targets. Imputed outcomes never affect any number.
#   * Threshold = 75 ug/m3 to match the project's event definition.
#     Set EVENT_THRESHOLD_1H = 76.0 for a strict comparison to TransNet.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import precision_recall_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
SEED = 0
EVENT_THRESHOLD_1H = 76.0
MAX_EPOCHS_P1 = 15
PATIENCE = 5
WINDOW = 36
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_causal_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

# True where the PM2.5 value at that station-hour was actually measured
target_obs_arr = (df.pivot(index="Datetime", columns="Station_ID", values="PM25_src")[station_order]
                    .reindex(dt_index).fillna("missing").to_numpy() == "observed")
print(f"PM2.5 observed share: {target_obs_arr.mean()*100:.2f}%")

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

def pinball_per_element(preds, target, quantiles):
    diff = target.unsqueeze(-1) - preds
    q = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(1, 1, -1)
    return torch.max(q * diff, (q - 1) * diff).mean(dim=-1)

def masked_pinball(preds, target, quantiles, mask):
    per_elem = pinball_per_element(preds, target, quantiles)
    return (per_elem * mask).sum() / mask.sum().clamp(min=1)

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
GRAPH_RECENT_HOURS = 18
print(f"device={DEVICE}")

edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
del train_nonzero, station_wind_raw
gc.collect()
wind_ewt_s = torch.tensor(station_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

# Next-hour windowing: input = t..t+35, target = t+36, both in the same split.
p_buckets = {0: ([], [], [], [], []), 1: ([], [], [], [], []), 2: ([], [], [], [], [])}
for t in range(0, n_time - WINDOW):
    target_t = t + WINDOW
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    Xl, yregl, yrawl, obsl, sl = p_buckets[s_start]
    Xl.append(time_arr_std[t:t + WINDOW])
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])   # standardized, imputed where unobserved
    yrawl.append(pm25_raw_arr[target_t, :])                 # raw, imputed where unobserved
    obsl.append(target_obs_arr[target_t, :])                # True only where the target was measured
    sl.append(t)

X0, yreg0, yraw0, obs0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, yraw1, obs1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, yraw2, obs2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")
print(f"observed targets: train={obs0.mean()*100:.1f}%  val={obs1.mean()*100:.1f}%  test={obs2.mean()*100:.1f}%")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t = torch.tensor(yreg0, dtype=torch.float32)
yreg1_t = torch.tensor(yreg1, dtype=torch.float32)
obs0_t = torch.tensor(obs0, dtype=torch.float32)
obs1_t = torch.tensor(obs1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

def run_p1_epoch(model, X, y, obs, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ob = obs[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = masked_pinball(pred, yb, QUANTILES, ob)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def predict_quantiles(model, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_stations, N_QUANTILES), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_seq = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            pred = model(xb, edge_index, ew_seq)
            out[idx.numpy()] = pred.cpu().numpy()
    return out

print("\n=== Phase 1: station-level next-hour quantile model (wind_graph, seed=0) ===")
torch.manual_seed(SEED); np.random.seed(SEED)
lr, dropout, wd = 0.003, 0.5, 0.0005
model = StationQuantileGCN(n_feats, dropout).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
best_val, best_state, patience_ctr, best_epoch = float("inf"), None, 0, 0
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    tl = run_p1_epoch(model, X0_t, yreg0_t, obs0_t, starts0_t, opt, True)
    vl = run_p1_epoch(model, X1_t, yreg1_t, obs1_t, starts1_t, opt, False)
    print(f"  epoch {epoch}  train_pinball={tl:.4f}  val_pinball={vl:.4f}")
    if vl < best_val: best_val, best_state, patience_ctr, best_epoch = vl, copy.deepcopy(model.state_dict()), 0, epoch
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE: break
model.load_state_dict(best_state)
print(f"best_epoch={best_epoch} val_pinball={best_val:.4f}")

val_quantiles = predict_quantiles(model, X1_t, starts1_t)
test_quantiles = predict_quantiles(model, X2_t, starts2_t)

pm25_mean_feat = t_mean[0, 0, pm25_col_idx]
pm25_std_feat = t_std[0, 0, pm25_col_idx]
val_quantiles_raw = val_quantiles * pm25_std_feat + pm25_mean_feat
test_quantiles_raw = test_quantiles * pm25_std_feat + pm25_mean_feat

# Labels and masks: evaluation uses ONLY station-hours with an observed target
true_val = (yraw1 >= EVENT_THRESHOLD_1H).astype(int)
true_test = (yraw2 >= EVENT_THRESHOLD_1H).astype(int)

print(f"\n{'='*20} selecting operating point on VALIDATION set (observed targets only) {'='*20}")
best_f1, best_q_idx, best_cutoff = -1, None, None
for q_idx, q in enumerate(QUANTILES):
    score_val = val_quantiles_raw[:, :, q_idx][obs1]
    y_val_obs = true_val[obs1]
    prec, rec, thresh = precision_recall_curve(y_val_obs, score_val)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    idx = np.nanargmax(f1[:-1])
    print(f"  q={q:.2f}: best val F1={f1[idx]:.4f} at cutoff={thresh[idx]:.2f} ug/m3")
    if f1[idx] > best_f1:
        best_f1, best_q_idx, best_cutoff = f1[idx], q_idx, thresh[idx]

print(f"\nselected: quantile={QUANTILES[best_q_idx]:.2f}  cutoff={best_cutoff:.2f} ug/m3  (val F1={best_f1:.4f})")

score_test = test_quantiles_raw[:, :, best_q_idx][obs2]
true_flat = true_test[obs2]
pred_flat = (score_test >= best_cutoff).astype(int)

TP = int(((pred_flat == 1) & (true_flat == 1)).sum())
FP = int(((pred_flat == 1) & (true_flat == 0)).sum())
FN = int(((pred_flat == 0) & (true_flat == 1)).sum())
TN = int(((pred_flat == 0) & (true_flat == 0)).sum())
n_total_events = TP + FN
recall = TP / max(n_total_events, 1)
precision = TP / max(TP + FP, 1)
event_rate = n_total_events / len(true_flat)

print(f"\n{'='*20} STATION-LEVEL +1h severe-event detection (test, observed targets only) {'='*20}")
print(f"test set: {len(true_flat)} observed station-hours, {n_total_events} events ({event_rate*100:.2f}%, threshold>={EVENT_THRESHOLD_1H})")
print(f"our model:  recall={recall*100:.1f}%  ({TP}/{n_total_events} detected)   "
      f"precision={precision*100:.1f}%   false alarms={FP}")

print(f"\n{'='*20} comparison to published +1h results {'='*20}")
print(f"{'model':15s} {'recall':>10s} {'detected':>14s} {'precision':>11s} {'false alarms':>13s}")
print(f"{'AGATNet':15s} {'84.5%':>10s} {'16079/17921':>14s} {'72.6%':>11s} {'6077':>13s}")
print(f"{'TransNet':15s} {'53.9%':>10s} {'9655/17921':>14s} {'94.0%':>11s} {'619':>13s}")
print(f"{'our model':15s} {f'{recall*100:.1f}%':>10s} {f'{TP}/{n_total_events}':>14s} {f'{precision*100:.1f}%':>11s} {f'{FP}':>13s}")
print(f"\n(their event rate: 1.2% at >=76 ug/m3; ours: {event_rate*100:.2f}% at >={EVENT_THRESHOLD_1H} ug/m3, observed targets only.)")

results = {
    "level": "station",
    "leak_controls": ["causal imputation", "observed-only targets in loss", "observed-only evaluation", "train-only normalization"],
    "threshold": EVENT_THRESHOLD_1H,
    "selected_quantile": QUANTILES[best_q_idx],
    "selected_cutoff_ugm3": float(best_cutoff),
    "n_test_station_hours_observed": int(len(true_flat)),
    "n_severe_events": int(n_total_events),
    "event_rate_pct": float(event_rate * 100),
    "TP": TP, "FP": FP, "FN": FN, "TN": TN,
    "recall_pct": float(recall * 100),
    "precision_pct": float(precision * 100),
}
with open(f"{BASE}/plus1h_station_level_comparison_causal_2019_seed0.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nsaved to {BASE}/plus1h_station_level_comparison_causal_2019_seed0.json")


PM2.5 observed share: 92.79%
device=mps
windows: train=17508 val=8724 test=8724
observed targets: train=89.1%  val=94.6%  test=95.7%

=== Phase 1: station-level next-hour quantile model (wind_graph, seed=0) ===
  epoch 1  train_pinball=0.1074  val_pinball=0.0685
  epoch 2  train_pinball=0.0876  val_pinball=0.0670
  epoch 3  train_pinball=0.0857  val_pinball=0.0669
  epoch 4  train_pinball=0.0848  val_pinball=0.0670
  epoch 5  train_pinball=0.0842  val_pinball=0.0688
  epoch 6  train_pinball=0.0831  val_pinball=0.0654
  epoch 7  train_pinball=0.0826  val_pinball=0.0654
  epoch 8  train_pinball=0.0819  val_pinball=0.0636
  epoch 9  train_pinball=0.0813  val_pinball=0.0662
